# 07 — Parallel heads

One pooled vector, three linears, three answers. They do not wait on each other. That is the "parallel sampler" idea at a size you can train: every question is a head, and they all read the same state.

- **Choice** — softmax, argmax name, confidence
- **Score** — softmax over ordered levels, score = sum of `index * probability`
- **Noul** — sigmoid

Temperature divides the logits before softmax. `1` leaves them alone.

**You implement:** `choice_answer`, `score_answer`, `noul_answer`.



In [ ]:
from __future__ import annotations

import torch
from torch import Tensor


def choice_answer(logits: Tensor, names: list[str], temperature: float = 1.0) -> dict:
    raise NotImplementedError("choice_answer")


def score_answer(logits: Tensor, legend: list[str], temperature: float = 1.0) -> dict:
    raise NotImplementedError("score_answer")


def noul_answer(logit: Tensor) -> dict:
    raise NotImplementedError("noul_answer")



<details>
<summary><b>Solution</b> (try yourself first — click to expand)</summary>

```python
import math
import torch
from torch import Tensor

def _confidence(probs: Tensor) -> float:
    k = probs.numel()
    entropy = -(probs.clamp_min(1e-8) * probs.clamp_min(1e-8).log()).sum()
    return float((1.0 - entropy / math.log(k)).clamp(0.0, 1.0))

def choice_answer(logits: Tensor, names: list[str], temperature: float = 1.0) -> dict:
    probs = torch.softmax(logits.float() / max(temperature, 1e-3), dim=-1)
    index = int(torch.argmax(probs))
    return {
        "choice": names[index],
        "probabilities": {name: float(probs[i]) for i, name in enumerate(names)},
        "confidence": _confidence(probs),
    }

def score_answer(logits: Tensor, legend: list[str], temperature: float = 1.0) -> dict:
    probs = torch.softmax(logits.float() / max(temperature, 1e-3), dim=-1)
    levels = torch.arange(probs.numel(), dtype=probs.dtype)
    return {
        "score": float((probs * levels).sum()),
        "probabilities": {str(i): float(probs[i]) for i in range(probs.numel())},
        "legend": list(legend),
        "confidence": _confidence(probs),
    }

def noul_answer(logit: Tensor) -> dict:
    return {"noul": float(torch.sigmoid(logit.float()))}
```

Copy into the stub cell above, then run **Check**.

</details>



## Check



In [ ]:
choice = choice_answer(torch.tensor([0.0, 5.0, 0.0]), ["chat", "ops", "learning"])
assert choice["choice"] == "ops"
assert abs(sum(choice["probabilities"].values()) - 1) < 1e-4
score = score_answer(torch.tensor([0.0, 0.0, 8.0]), ["calm", "soon", "now"])
assert score["score"] > 1.9
reply = noul_answer(torch.tensor(0.0))
assert abs(reply["noul"] - 0.5) < 1e-5
print(choice["choice"], round(score["score"], 3), reply["noul"])

